# 07. Qwen LoRA → GGUF 변환

**목적:** 파인튜닝한 Qwen 모델을 Ollama에서 사용할 수 있도록 GGUF 형식으로 변환

**환경:** Google Colab (T4 GPU 권장)

**예상 시간:** 30분~1시간

In [1]:
# 셀 1: Kaggle 환경 확인
import os
print("✅ Kaggle 환경 확인 완료")
print(f"GPU: {os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()}")

✅ Kaggle 환경 확인 완료
GPU: Tesla T4
Tesla T4


In [2]:
# 셀 2: 필요 라이브러리 설치
!pip install torch transformers peft accelerate bitsandbytes -q
print("✅ 라이브러리 설치 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 33.4 MB/s eta 0:00:00:00:0100:01
✅ 라이브러리 설치 완료


In [3]:
# 셀 3: LoRA 어댑터 확인
import os

# Kaggle 경로
LORA_PATH = "/kaggle/input/civillora"

if os.path.exists(LORA_PATH):
    print("✅ LoRA 어댑터 발견!")
    print("파일 목록:")
    for f in os.listdir(LORA_PATH):
        print(f"  - {f}")
else:
    print("❌ LoRA 어댑터를 찾을 수 없습니다. 경로를 확인하세요.")

✅ LoRA 어댑터 발견!
파일 목록:
  - adapter_model.safetensors
  - merges.txt
  - adapter_config.json
  - tokenizer.json
  - tokenizer_config.json
  - chat_template.jinja
  - special_tokens_map.json
  - added_tokens.json


In [11]:
# 셀: Base + LoRA → GGUF 직접 변환
print("="*60)
print("🔄 LoRA → GGUF 직접 변환")
print("="*60)

# 1. Base 모델 다운로드
from huggingface_hub import snapshot_download

print("📥 Base 모델 다운로드 중...")
base_path = snapshot_download(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    local_dir="/kaggle/working/qwen-base",
    ignore_patterns=["*.bin"]  # safetensors만
)
print(f"✅ Base 모델: {base_path}")

# 2. LoRA → GGUF 변환
print("\n🔄 LoRA → GGUF 변환 중...")
!python /kaggle/working/llama.cpp/convert_lora_to_gguf.py \
    /kaggle/input/civillora \
    --outfile /kaggle/working/civil-lora.gguf \
    --base /kaggle/working/qwen-base

print("\n✅ 완료!")
!ls -lh /kaggle/working/*.gguf

🔄 LoRA → GGUF 직접 변환
📥 Base 모델 다운로드 중...


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Base 모델: /kaggle/working/qwen-base

🔄 LoRA → GGUF 변환 중...
INFO:lora-to-gguf:Loading base model: qwen-base
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00004.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:lora-to-gguf:Exporting model...
INFO:hf-to-gguf:blk.0.attn_q.weight.lora_a, torch.float32 --> F32, shape = {3584, 16}
INFO:hf-to-gguf:blk.0.attn_q.weight.lora_b, torch.float32 --> F32, shape = {16, 3584}
INFO:hf-to-gguf:blk.0.attn_v.weight.lora_a, torch.float32 --> F32, shape = {3584, 16}
INFO:hf-to-gguf:blk.0.attn_v.weight.lora_b, torch.float32 --> F32, shape = {16, 512}
INFO:hf-to-gguf:blk.1.attn_q.weight.lora_a, torch.fl

In [14]:
# 셀 6: 변환된 파일 확인
import os

GGUF_FILE = "/kaggle/working/civil-lora.gguf"

if os.path.exists(GGUF_FILE):
    size_mb = os.path.getsize(GGUF_FILE) / (1024**2)
    print(f"✅ GGUF 파일 생성됨: {size_mb:.2f} MB")
    print(f"\n📁 Output 탭에서 다운로드하세요!")
else:
    print("❌ GGUF 파일을 찾을 수 없습니다.")


✅ GGUF 파일 생성됨: 19.26 MB

📁 Output 탭에서 다운로드하세요!


In [15]:
# 셀 7: Output에서 다운로드
print("="*60)
print("📥 다운로드")
print("="*60)
print("\nKaggle Output 탭에서 civil-lora.gguf 파일을 다운로드하세요!")
print("파일 크기: 약 20MB")
print("\n로컬 저장 경로: ~/CIVILCOMPLAINT/models/llm_qwen/civil-lora.gguf")

📥 다운로드

Kaggle Output 탭에서 civil-qwen-q4.gguf 파일을 다운로드하세요!
파일 크기: 약 4GB


---
## 로컬에서 Ollama 등록 방법

### 1. GGUF 파일 다운로드 후 로컬에 저장
```
~/CIVILCOMPLAINT/models/llm_qwen/civil-lora.gguf
```

### 2. Base 모델 다운로드
```bash
ollama pull qwen2.5:7b-instruct
```

### 3. Modelfile 생성
```bash
cat > ~/CIVILCOMPLAINT/models/llm_qwen/Modelfile << 'EOF'
FROM qwen2.5:7b-instruct
ADAPTER ./civil-lora.gguf
SYSTEM "당신은 친절하고 전문적인 공공기관 민원 상담 AI입니다."
PARAMETER temperature 0.7
PARAMETER num_predict 512
EOF
```

### 4. Ollama에 등록
```bash
cd ~/CIVILCOMPLAINT/models/llm_qwen
ollama create civil-qwen -f Modelfile
```

### 5. 테스트
```bash
ollama run civil-qwen "인터넷뱅킹 비밀번호를 잊어버렸어요"
```